In [1]:
import pandas as pd
import numpy as np

import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.decision_tree import DecisionTree
from src.preprocessing import load_processed
from src.evaluation import cross_validation_loop

In [2]:
datasets = {
    "Breast Cancer": "../data/processed/breast_cancer_processed.csv",
    "Car Evaluation": "../data/processed/car_evaluation_processed.csv",
    "House Votes": "../data/processed/house_votes_processed.csv",
    "Abalone": "../data/processed/abalone_processed.csv",
    "Computer Hardware": "../data/processed/computer_hardware_processed.csv",
    "Forest Fires": "../data/processed/forest_fires_processed.csv"
}

target = {
    "Breast Cancer": "class",
    "Car Evaluation": "class",
    "House Votes": "class",
    "Abalone": "rings",
    "Computer Hardware": "prp",
    "Forest Fires": "area"
}

method = {
    "Breast Cancer": "classification",
    "Car Evaluation": "classification",
    "House Votes": "classification",
    "Abalone": "regression",
    "Computer Hardware": "regression",
    "Forest Fires": "regression"
}

In [7]:
seed = 0
rows = []

for dataset_name, dataset_path in datasets.items():
    print(f"Running experiments for {dataset_name} Dataset...")
    data, feature_types = load_processed(dataset_path)

    target_col = target[dataset_name]
    mode = method[dataset_name]

    feature_data = data.drop(columns=[target_col]).to_numpy()
    label_data = data[target_col].to_numpy()

    results = results = cross_validation_loop(feature_data, label_data, feature_types, mode=mode, seed=seed)

    rows.append({
        "Dataset": dataset_name,
        "Unpruned Error": np.mean(results["unpruned"]),
        "Pruned Error": np.mean(results["pruned"]),
        "Mean Unpruned Tree Size": np.mean(results["unpruned size"]),
        "Mean Pruned Tree Size": np.mean(results["pruned size"]),
    })

results_df = pd.DataFrame(rows).set_index("Dataset")
results_df = results_df.round({
    "Unpruned Error": 4,
    "Pruned Error": 4,
    "Mean Unpruned Tree Size": 2,
    "Mean Pruned Tree Size": 2,
})

results_df

Running experiments for Breast Cancer Dataset...
Running experiments for Car Evaluation Dataset...
Running experiments for House Votes Dataset...
Running experiments for Abalone Dataset...
Running experiments for Computer Hardware Dataset...
Running experiments for Forest Fires Dataset...


,Unpruned Error,Pruned Error,Mean Unpruned Tree Size,Mean Pruned Tree Size
Dataset,,,,
Breast Cancer,0.0505,0.0426,31.4,13.6
Car Evaluation,0.0443,0.0433,113.0,73.8
House Votes,0.0747,0.0500,27.5,6.1
Abalone,9.1133,6.1419,2002.8,315.3
Computer Hardware,9752.7891,9333.1269,153.4,42.4
Forest Fires,4.1381,2.2138,262.0,21.6


In [6]:
results_df.to_csv(f"../results/raw_scores/seed_{seed}_results.csv")